In [1]:

# Import Required Modules
from flask import Flask, render_template
import pandas as pd
import json
import plotly
import plotly.express as px
import pandas_ta as pta
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from datetime import date
from datetime import datetime
from datetime import timedelta, date
from prophet import Prophet
import pandas as pd
import numpy as np
import yfinance as yf
import seaborn as sns
import mpld3
import matplotlib.pyplot as plt
import warnings


# Create Home Page Route
app = Flask(__name__)


   
df = pd.read_csv("../Resources/btcjoin.csv", parse_dates=['date'])
btc_df = yf.Ticker('BTC-USD').history(period='7y',interval='1d',actions=False).reset_index()
btc_df = btc_df.loc[(btc_df['Date'] > '2022-10-25')]
btc_df['Close']=btc_df['Close'].astype("float")
df['price']=df['price'].str.replace(',','')
df['price']=df['price'].astype("float")
btc_df = btc_df.rename(columns={"Close": "price", "Date":"date"})
btc_df['date'] = btc_df['date'].apply(lambda x: x.strftime('%Y-%m-%d'))
btc_df['date'] = pd.to_datetime(btc_df['date'])

df = pd.merge(df, btc_df, on=['date', 'price'], how='outer')
df = pd.merge(df, btc_df, on=['date', 'price'], how='outer')
df = df.rename(columns={"value": "wallets"})
df = df.drop(columns=['volume','change', 'low', 'high', 'open','Unnamed: 0', "wallets", "address", "mined"])
df['200D'] = df['price'].rolling(200).mean()
df['300D'] = df['price'].rolling(300).mean()
df['50D'] = df['price'].rolling(50).mean()
df['7D'] = df['price'].rolling(7).mean()
# df = df.dropna()
df['meanavge'] = (df['200D'] + df['300D'] + df['50D'] )/3
# df = df.drop(columns=['200D','300D', '50D'])
df['meanvalue'] = df["price"] - df["meanavge"]
df['status'] = df['meanvalue'].apply(lambda x: '1' if x > 0 else '0')
df['status']=df['status'].astype("object")
df['price-meanavge']=df['price'] - df['meanavge']
df['move%'] = (df['price-meanavge']/(df['price'] + df['meanavge']))
bins = [-.43, -.18, 0, .18, .43]
group_names = ["Severely Oversold","Neutral Oversold", "Neutral Overbought","Severely Overbought"]
df["Valuation"] = pd.cut(df["move%"], bins, labels=group_names)

k = df['price'].ewm(span=12, adjust=False, min_periods=12).mean()

# Get the 12-day EMA of the closing price
d = df['price'].ewm(span=26, adjust=False, min_periods=26).mean()

# Subtract the 26-day EMA from the 12-Day EMA to get the MACD
macd = k - d

# Get the 9-Day EMA of the MACD for the Trigger line
macd_s = macd.ewm(span=9, adjust=False, min_periods=9).mean()

# Calculate the difference between the MACD - Trigger for the Convergence/Divergence value
macd_h = macd - macd_s

# Add all of our new values for the MACD to the dataframe
df['macd'] = df.index.map(macd)
df['macd_h'] = df.index.map(macd_h)
df['macd_s'] = df.index.map(macd_s)


df['priceL'] = np.log(df['price'])

df_train = df[['date', 'priceL']]
df_train = df_train.rename(columns = {"date":"ds", "priceL":"y"})

# instantiate the model and set parameters
model = Prophet()

# fit the model to historical data
model.fit(df_train);

start = "2010-09-25"
end = date.today() + timedelta(days=60)
insample = pd.DataFrame(pd.date_range(start,end, periods=92))

# Change the column name
insample.columns = ['ds']

# in-sample prediction
prediction = model.predict(insample)


# Before calling the function or method that triggers the warning:
with warnings.catch_warnings():
    # Ignore FutureWarning raised within this block
    warnings.simplefilter("ignore", FutureWarning)

# Buy Zones
    fig = px.scatter(df, x="date", y="price", color="Valuation", color_discrete_sequence=["red","green","blue","orange"],
                    title="price")
    fig.add_trace(go.Scatter(name="MeanAvg", x=df['date'], y=df['meanavge'], marker = {'color' : 'black'}, legendrank=2))
    fig.add_trace(go.Scatter(x=prediction['ds'], y=np.exp(prediction['yhat']),
        fill=None,
        mode='lines',
        line_color='lightblue',
        ))
    fig.add_trace(go.Scatter(
        x=prediction['ds'],
        y=np.exp(prediction['yhat_lower']),
        fill='tonexty', # fill area between trace0 and trace1
        mode='lines', line_color='lightblue'))

    fig.add_trace(go.Scatter(
        x=prediction['ds'],
        y=np.exp(prediction['yhat_upper']),
        fill='tonexty', # fill area between trace0 and trace1
        mode='lines', line_color='lightblue'))

fig.update_yaxes(fixedrange=False)
fig.update_layout(title_text='Bitcoin Prophet Model + Buy Zones')
fig.update_yaxes(type="log")
fig.update_xaxes(ticklabelposition="inside top", title="Date")
fig.update_yaxes(nticks=12)
fig.update_xaxes(nticks=50)
fig.update_layout(
    margin=dict(l=20, r=100, t=70, b=20),
)
fig.update_layout(height=500, width=1000)
fig.update_layout(showlegend=False)
fig.add_vline(x='2012-11-28', line_width=3, line_dash="dash", line_color="green")
fig.add_vline(x='2016-07-09', line_width=3, line_dash="dash", line_color="green")	
fig.add_vline(x='2020-05-11', line_width=3, line_dash="dash", line_color="green")	
fig.add_vline(x='2024-04-02', line_width=3, line_dash="dash", line_color="green")
fig.update_layout(template='plotly_white')
fig.show()



06:13:58 - cmdstanpy - INFO - Chain [1] start processing
06:14:02 - cmdstanpy - INFO - Chain [1] done processing


In [6]:
# Define the Dark Theme Color Palette
BG_COLOR = '#0e1117'
GRID_COLOR = '#262730'
# High-contrast colors for dark mode
VALUATION_COLORS = ["#ff3131", "#39ff14", "#00d4ff", "#ffae42"] # Neon Red, Green, Blue, Orange

# Buy Zones Base Scatter
fig = px.scatter(df, x="date", y="price", 
                 color="Valuation", 
                 color_discrete_sequence=VALUATION_COLORS,
                 title="price")

# Mean Average - Switched from Black to White/Light Gray for visibility
fig.add_trace(go.Scatter(name="MeanAvg", x=df['date'], y=df['meanavge'], 
                         line=dict(color='white', width=1.5, dash='dot'), 
                         legendrank=2))

# Prophet Trend Line
fig.add_trace(go.Scatter(x=prediction['ds'], y=np.exp(prediction['yhat']),
    fill=None,
    mode='lines',
    line_color='#00d4ff', # Electric Blue
    name="Prophet Trend"
    ))

# Prophet Confidence Intervals (Lower)
fig.add_trace(go.Scatter(
    x=prediction['ds'],
    y=np.exp(prediction['yhat_lower']),
    fill=None, # Base for the fill
    mode='lines', 
    line_color='rgba(0, 212, 255, 0)', # Invisible line
    showlegend=False))

# Prophet Confidence Intervals (Upper + Fill)
fig.add_trace(go.Scatter(
    x=prediction['ds'],
    y=np.exp(prediction['yhat_upper']),
    fill='tonexty', 
    fillcolor='rgba(0, 212, 255, 0.15)', # Subtle blue glow
    mode='lines', 
    line_color='rgba(0, 212, 255, 0)', # Invisible line
    showlegend=False))

# Layout & Theme Adjustments
fig.update_layout(
    template='plotly_dark',
    paper_bgcolor=BG_COLOR,
    plot_bgcolor=BG_COLOR,
    title_text='<b>BITCOIN PROPHET MODEL + BUY ZONES</b>',
    title_font=dict(size=20, color='white', family="Courier New"),
    height=600, 
    width=1100,
    showlegend=False,
    margin=dict(l=50, r=50, t=100, b=50)
)

# Axis Customization
fig.update_yaxes(
    type="log",
    fixedrange=False,
    nticks=12,
    gridcolor=GRID_COLOR,
    zerolinecolor=GRID_COLOR,
    title="Price (USD)"
)

fig.update_xaxes(
    type="date",
    nticks=20,
    gridcolor=GRID_COLOR,
    zerolinecolor=GRID_COLOR,
    title="Timeline"
)

# Halving Lines - Adjusted opacity so they don't clash with price dots
halving_dates = ['2012-11-28', '2016-07-09', '2020-05-11', '2024-04-20']
for h_date in halving_dates:
    fig.add_vline(x=h_date, line_width=2, line_dash="dash", line_color="rgba(57, 255, 20, 0.4)")

fig.show()

In [19]:
# --- 1. Data Prep ---
df_btc = df[['date', 'price']].copy().rename(columns={'price': 'btc_price'})
df_btc['date'] = pd.to_datetime(df_btc['date'])
df_btc = df_btc.set_index('date')

# Fetching Gold Spot (XAUUSD=X) for the most accurate currency ratio
gold_df = yf.Ticker('GC=F').history(period='max')[['Close']].reset_index()
gold_df = gold_df.rename(columns={'Close': 'gold_price', 'Date': 'date'})
gold_df['date'] = pd.to_datetime(gold_df['date']).dt.tz_localize(None)
gold_df = gold_df.set_index('date')

ratio_df = df_btc.join(gold_df, how='inner')
ratio_df['ratio'] = ratio_df['btc_price'] / ratio_df['gold_price']

# Indicators
ratio_df['MA50'] = ratio_df['ratio'].rolling(50).mean()
ratio_df['MA200'] = ratio_df['ratio'].rolling(200).mean()
ath_ratio = ratio_df['ratio'].max()
ath_date = ratio_df['ratio'].idxmax()

# --- 2. Visualization ---
fig = go.Figure()

# Main Ratio Area
fig.add_trace(go.Scatter(
    x=ratio_df.index, 
    y=ratio_df['ratio'],
    fill='tozeroy', 
    name='Ounces of Gold per BTC',
    line=dict(color='#00ffce', width=2), # Neon cyan/mint for high contrast
    fillcolor='rgba(0, 255, 206, 0.05)'
))

# 50-Day Moving Average
fig.add_trace(go.Scatter(
    x=ratio_df.index, 
    y=ratio_df['MA50'],
    name='50D Moving Avg',
    line=dict(color='rgba(255, 255, 255, 0.4)', width=1.5, dash='dot')
))

# 200-Day Moving Average
fig.add_trace(go.Scatter(
    x=ratio_df.index,
    y=ratio_df['MA200'],
    name='200D Moving Avg',
    mode='lines',
    line=dict(color='rgba(255, 174, 66, 0.9)', width=1)
))

# Annotation for All-Time High
fig.add_annotation(
    x=ath_date,
    y=ath_ratio,
    text=f"ATH: {ath_ratio:.2f} oz",
    showarrow=True,
    arrowhead=2,
    arrowcolor="#FFD700",
    ax=0,
    ay=-40,
    font=dict(color="#FFD700", size=12)
)

# Layout Configuration (Linear Scale)
fig.update_layout(
    title=dict(
        text='<b>Bitcoin vs. Gold Ratio (Linear Scale)</b><br><sup>The purchasing power of 1 BTC denominated in Ounces of Gold</sup>',
        font=dict(size=20, color='white'),
        x=0.05
    ),
    template='plotly_dark',
    paper_bgcolor='#0e1117',
    plot_bgcolor='#0e1117',
    yaxis=dict(
        type="linear", # Explicitly set to linear
        title="Gold Ounces",
        side="right",
        gridcolor='rgba(255, 255, 255, 0.05)',
        zeroline=False
    ),
    xaxis=dict(
        title="",
        gridcolor='rgba(255, 255, 255, 0.05)',
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1M", step="month", stepmode="backward"),
                dict(count=3, label="3M", step="month", stepmode="backward"),
                dict(count=6, label="6M", step="month", stepmode="backward"),
                dict(count=1, label="YTD", step="year", stepmode="todate"),
                dict(count=1, label="1Y", step="year", stepmode="backward"),
                dict(count=3, label="3Y", step="year", stepmode="backward"),
                dict(count=5, label="5Y", step="year", stepmode="backward"),
                dict(step="all", label="MAX")
            ]),
            bgcolor='#1e2130'
        )
    ),
    hovermode="x unified",
    margin=dict(l=30, r=60, t=100, b=40),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig.update_layout(height=500, width=1000)
fig.show()

In [13]:
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go

# --- 1. Data Prep ---
mstr_ticker = yf.Ticker('MSTR')
mstr_df = mstr_ticker.history(period='max')[['Close']].reset_index()
mstr_df = mstr_df.rename(columns={'Close': 'mstr_price', 'Date': 'date'})
mstr_df['date'] = pd.to_datetime(mstr_df['date']).dt.tz_localize(None)
mstr_df = mstr_df.set_index('date')

gold_ticker = yf.Ticker('GC=F')
gold_df = gold_ticker.history(period='max')[['Close']].reset_index()
gold_df = gold_df.rename(columns={'Close': 'gold_price', 'Date': 'date'})
gold_df['date'] = pd.to_datetime(gold_df['date']).dt.tz_localize(None)
gold_df = gold_df.set_index('date')

ratio_df = mstr_df.join(gold_df, how='inner')
ratio_df['ratio'] = ratio_df['mstr_price'] / ratio_df['gold_price']

# Indicators
ratio_df['MA50'] = ratio_df['ratio'].rolling(50).mean()
ath_ratio = ratio_df['ratio'].max()
ath_date = ratio_df['ratio'].idxmax()

# --- 2. Visualization ---
fig = go.Figure()

# Main Ratio Area
fig.add_trace(go.Scatter(
    x=ratio_df.index, 
    y=ratio_df['ratio'],
    fill='tozeroy', 
    name='Ounces of Gold per MSTR Share',
    line=dict(color='#ff7b00', width=2), 
    fillcolor='rgba(255, 123, 0, 0.05)'
))

# 50-Day Moving Average
fig.add_trace(go.Scatter(
    x=ratio_df.index, 
    y=ratio_df['MA50'],
    name='50D Trend',
    line=dict(color='rgba(255, 255, 255, 0.3)', width=1.5, dash='dot')
))

# ATH Annotation
fig.add_annotation(
    x=ath_date,
    y=ath_ratio,
    text=f"ATH Ratio: {ath_ratio:.3f} oz",
    showarrow=True,
    arrowhead=2,
    ax=0,
    ay=-40,
    font=dict(color="#ff7b00", size=12)
)

# Layout Configuration
fig.update_layout(
    title=dict(
        text='<b>MicroStrategy vs. Gold Ratio (MSTR/XAU)</b><br><sup>The purchasing power of 1 MSTR share in Ounces of Gold</sup>',
        font=dict(size=20, color='white'),
        x=0.05
    ),
    template='plotly_dark',
    paper_bgcolor='#0e1117',
    plot_bgcolor='#0e1117',
    yaxis=dict(
        type="linear", 
        title="Gold Ounces per Share",
        side="right",
        gridcolor='rgba(255, 255, 255, 0.05)',
        tickformat=".3f"
    ),
    xaxis=dict(
        gridcolor='rgba(255, 255, 255, 0.05)',
        type='date',
        # --- UPDATED RANGE SELECTOR ---
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1M", step="month", stepmode="backward"),
                dict(count=3, label="3M", step="month", stepmode="backward"),
                dict(count=6, label="6M", step="month", stepmode="backward"),
                dict(count=1, label="YTD", step="year", stepmode="todate"),
                dict(count=1, label="1Y", step="year", stepmode="backward"),
                dict(count=3, label="3Y", step="year", stepmode="backward"),
                dict(count=5, label="5Y", step="year", stepmode="backward"),
                dict(step="all", label="MAX")
            ]),
            bgcolor='#1e2130',
            activecolor='#ff7b00',
            font=dict(size=11),
            y=1. # Adjusted position so it doesn't hit the title
        )
    ),
    hovermode="x unified",
    margin=dict(l=30, r=60, t=120, b=40), # Increased top margin for buttons
    legend=dict(orientation="h", yanchor="bottom", y=1.0, xanchor="right", x=1)
)
fig.update_layout(height=500, width=1000)
fig.show()